In [1]:
"""
Breast & Brain Cancer Detection System
Medical Image Analysis using Deep Learning
Author: Medical AI Research Team
"""

# ============================================================================
# IMPORTS AND SETUP
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import warnings
from pathlib import Path

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                            accuracy_score, precision_score, recall_score,
                            f1_score, roc_curve, auc, precision_recall_fscore_support)
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE, ADASYN

# Outlier Detection
from sklearn.ensemble import IsolationForest
from scipy import stats

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print("="*70)
print(" MEDICAL IMAGE ANALYSIS FOR CANCER DETECTION")
print(" Deep Learning Based Diagnostic Support System")
print("="*70)

# ============================================================================
# DATA PREPROCESSING PIPELINE
# ============================================================================

class MedicalImageProcessor:
    """
    Comprehensive medical image preprocessing system following clinical standards
    """

    def __init__(self, data_directory, image_dimensions=(224, 224)):
        self.data_dir = data_directory
        self.img_size = image_dimensions
        self.images = []
        self.labels = []
        self.class_names = []
        self.data_info = {}

    def load_dataset(self):
        """
        STEP 1: LOAD AND EXPLORE DATASET
        Loads medical images and performs initial data exploration
        """
        print("\n" + "="*70)
        print("STEP 1: LOADING AND EXPLORING DATASET")
        print("="*70)

        if not os.path.exists(self.data_dir):
            raise FileNotFoundError(f"Data directory not found: {self.data_dir}")

        self.class_names = sorted(os.listdir(self.data_dir))
        print(f"\nDetected Classes: {self.class_names}")

        class_distribution = {}
        total_images = 0

        for class_idx, class_name in enumerate(self.class_names):
            class_path = os.path.join(self.data_dir, class_name)

            if not os.path.isdir(class_path):
                continue

            image_files = [f for f in os.listdir(class_path)
                          if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]

            class_distribution[class_name] = len(image_files)

            for img_name in image_files:
                img_path = os.path.join(class_path, img_name)
                try:
                    # Load and preprocess image
                    img = cv2.imread(img_path)
                    if img is not None:
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        img = cv2.resize(img, self.img_size)
                        self.images.append(img)
                        self.labels.append(class_idx)
                        total_images += 1
                except Exception as e:
                    print(f"Warning: Could not load {img_path}: {str(e)}")
                    continue

        self.images = np.array(self.images, dtype=np.uint8)
        self.labels = np.array(self.labels)

        # Data exploration
        print(f"\nDataset Summary:")
        print(f"  Total Images: {len(self.images)}")
        print(f"  Image Shape: {self.images.shape}")
        print(f"  Data Type: {self.images.dtype}")
        print(f"  Memory Usage: {self.images.nbytes / (1024**2):.2f} MB")

        print(f"\nClass Distribution:")
        for class_name, count in class_distribution.items():
            percentage = (count / total_images) * 100
            print(f"  {class_name:20s}: {count:5d} images ({percentage:5.2f}%)")

        # Statistical summary
        print(f"\nPixel Intensity Statistics:")
        print(f"  Mean: {np.mean(self.images):.2f}")
        print(f"  Std Dev: {np.std(self.images):.2f}")
        print(f"  Min: {np.min(self.images)}")
        print(f"  Max: {np.max(self.images)}")

        self.data_info['class_distribution'] = class_distribution
        self.data_info['total_images'] = total_images

        return self.images, self.labels

    def check_and_remove_duplicates(self):
        """
        STEP 2: CHECK FOR AND HANDLE DUPLICATES
        Identifies and removes duplicate medical images
        """
        print("\n" + "="*70)
        print("STEP 2: CHECKING FOR DUPLICATE IMAGES")
        print("="*70)

        initial_count = len(self.images)

        # Create hash for each image
        print("Computing image signatures...")
        image_hashes = []
        for img in self.images:
            # Use multiple hash methods for accuracy
            img_bytes = img.tobytes()
            hash_value = hash(img_bytes)
            image_hashes.append(hash_value)

        # Find duplicates
        unique_hashes, unique_indices = np.unique(image_hashes, return_index=True)

        duplicate_count = initial_count - len(unique_indices)

        print(f"\nDuplicate Analysis:")
        print(f"  Total Images: {initial_count}")
        print(f"  Unique Images: {len(unique_indices)}")
        print(f"  Duplicates Found: {duplicate_count}")
        print(f"  Duplicate Percentage: {(duplicate_count/initial_count)*100:.2f}%")

        if duplicate_count > 0:
            print(f"\nRemoving {duplicate_count} duplicate images...")
            self.images = self.images[unique_indices]
            self.labels = self.labels[unique_indices]
            print(f"Dataset size after deduplication: {len(self.images)} images")
        else:
            print("No duplicates found. Dataset is clean.")

        self.data_info['duplicates_removed'] = duplicate_count

        return self.images, self.labels

    def handle_missing_and_corrupted(self):
        """
        STEP 3: APPLY TECHNIQUES TO HANDLE MISSING VALUES
        Detects and handles missing or corrupted image data
        """
        print("\n" + "="*70)
        print("STEP 3: HANDLING MISSING AND CORRUPTED DATA")
        print("="*70)

        initial_count = len(self.images)
        valid_indices = []
        corrupted_count = 0

        print("Scanning for corrupted or invalid images...")

        for idx, img in enumerate(self.images):
            # Check for various corruption issues
            is_valid = True

            # Check if image is empty
            if img is None or img.size == 0:
                is_valid = False

            # Check for NaN values
            elif np.isnan(img).any():
                is_valid = False

            # Check if image has correct shape
            elif img.shape != (*self.img_size, 3):
                is_valid = False

            # Check if image has reasonable pixel values
            elif np.min(img) < 0 or np.max(img) > 255:
                is_valid = False

            if is_valid:
                valid_indices.append(idx)
            else:
                corrupted_count += 1

        print(f"\nData Integrity Check:")
        print(f"  Valid Images: {len(valid_indices)}")
        print(f"  Corrupted/Missing: {corrupted_count}")
        print(f"  Corruption Rate: {(corrupted_count/initial_count)*100:.2f}%")

        if corrupted_count > 0:
            print(f"\nRemoving {corrupted_count} corrupted images...")
            self.images = self.images[valid_indices]
            self.labels = self.labels[valid_indices]
            print(f"Clean dataset size: {len(self.images)} images")
        else:
            print("All images passed integrity check.")

        self.data_info['corrupted_removed'] = corrupted_count

        return self.images, self.labels

    def detect_and_handle_outliers(self):
        """
        STEP 4: CHECK OUTLIERS AND HANDLE THEM
        Uses multiple methods to detect and handle outlier images
        """
        print("\n" + "="*70)
        print("STEP 4: DETECTING AND HANDLING OUTLIERS")
        print("="*70)

        # Calculate multiple features for outlier detection
        mean_intensities = np.mean(self.images, axis=(1, 2, 3))
        std_intensities = np.std(self.images, axis=(1, 2, 3))
        brightness = np.mean(self.images, axis=(1, 2, 3))

        print("Computing statistical measures...")

        # Method 1: Z-Score
        z_scores = np.abs(stats.zscore(mean_intensities))
        z_outliers = np.where(z_scores > 3)[0]

        # Method 2: IQR (Interquartile Range)
        Q1 = np.percentile(mean_intensities, 25)
        Q3 = np.percentile(mean_intensities, 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        iqr_outliers = np.where((mean_intensities < lower_bound) |
                                (mean_intensities > upper_bound))[0]

        # Method 3: Isolation Forest
        features = np.column_stack([mean_intensities, std_intensities])
        iso_forest = IsolationForest(contamination=0.05, random_state=42)
        iso_predictions = iso_forest.fit_predict(features)
        iso_outliers = np.where(iso_predictions == -1)[0]

        # Combine outlier detection methods
        all_outliers = np.unique(np.concatenate([z_outliers, iqr_outliers, iso_outliers]))

        print(f"\nOutlier Detection Results:")
        print(f"  Z-Score Method: {len(z_outliers)} outliers")
        print(f"  IQR Method: {len(iqr_outliers)} outliers")
        print(f"  Isolation Forest: {len(iso_outliers)} outliers")
        print(f"  Combined Total: {len(all_outliers)} outliers")
        print(f"  Outlier Percentage: {(len(all_outliers)/len(self.images))*100:.2f}%")

        print(f"\nStatistical Boundaries:")
        print(f"  Mean Intensity - Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
        print(f"  Lower Bound: {lower_bound:.2f}")
        print(f"  Upper Bound: {upper_bound:.2f}")

        # Decision on handling outliers
        outlier_percentage = (len(all_outliers) / len(self.images)) * 100

        if outlier_percentage < 5:
            print(f"\nOutliers are less than 5%. Retaining all data for model robustness.")
            outliers_removed = 0
        else:
            print(f"\nOutliers exceed 5%. Removing extreme outliers...")
            # Remove only the most extreme outliers (top 5%)
            n_to_remove = int(len(self.images) * 0.05)
            extreme_outliers = all_outliers[:n_to_remove]

            valid_indices = np.setdiff1d(np.arange(len(self.images)), extreme_outliers)
            self.images = self.images[valid_indices]
            self.labels = self.labels[valid_indices]
            outliers_removed = n_to_remove
            print(f"Removed {outliers_removed} extreme outliers.")
            print(f"Remaining images: {len(self.images)}")

        self.data_info['outliers_detected'] = len(all_outliers)
        self.data_info['outliers_removed'] = outliers_removed
        self.data_info['mean_intensities'] = mean_intensities

        return self.images, self.labels, mean_intensities

    def create_comprehensive_visualizations(self):
        """
        STEP 5: MAKE VISUALIZATIONS TO CHECK PROBLEMS, DISTRIBUTION, AND BALANCE
        Creates minimum 5-7 visualizations for data analysis
        """
        print("\n" + "="*70)
        print("STEP 5: CREATING COMPREHENSIVE VISUALIZATIONS")
        print("="*70)

        fig = plt.figure(figsize=(20, 16))

        # Visualization 1: Class Distribution
        print("Creating visualization 1/7: Class Distribution...")
        ax1 = plt.subplot(3, 3, 1)
        unique, counts = np.unique(self.labels, return_counts=True)
        colors = plt.cm.Set3(np.linspace(0, 1, len(unique)))
        bars = ax1.bar([self.class_names[i] for i in unique], counts, color=colors, edgecolor='black', linewidth=1.5)
        ax1.set_title('Class Distribution', fontsize=14, fontweight='bold', pad=10)
        ax1.set_xlabel('Class', fontsize=11, fontweight='bold')
        ax1.set_ylabel('Number of Images', fontsize=11, fontweight='bold')
        ax1.tick_params(axis='x', rotation=45)

        # Add count labels on bars
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontweight='bold')

        # Visualization 2: Pixel Intensity Distribution
        print("Creating visualization 2/7: Pixel Intensity Distribution...")
        ax2 = plt.subplot(3, 3, 2)
        mean_intensities = np.mean(self.images, axis=(1, 2, 3))
        ax2.hist(mean_intensities, bins=50, color='#FF6B6B', edgecolor='black', alpha=0.7)
        ax2.axvline(np.mean(mean_intensities), color='darkred', linestyle='--', linewidth=2, label=f'Mean: {np.mean(mean_intensities):.2f}')
        ax2.axvline(np.median(mean_intensities), color='blue', linestyle='--', linewidth=2, label=f'Median: {np.median(mean_intensities):.2f}')
        ax2.set_title('Pixel Intensity Distribution', fontsize=14, fontweight='bold', pad=10)
        ax2.set_xlabel('Mean Pixel Intensity', fontsize=11, fontweight='bold')
        ax2.set_ylabel('Frequency', fontsize=11, fontweight='bold')
        ax2.legend()
        ax2.grid(axis='y', alpha=0.3)

        # Visualization 3: Outlier Detection (Boxplot)
        print("Creating visualization 3/7: Outlier Detection...")
        ax3 = plt.subplot(3, 3, 3)
        bp = ax3.boxplot(mean_intensities, vert=True, patch_artist=True)
        bp['boxes'][0].set_facecolor('#4ECDC4')
        bp['boxes'][0].set_edgecolor('black')
        bp['boxes'][0].set_linewidth(2)
        for whisker in bp['whiskers']:
            whisker.set(linewidth=2)
        for cap in bp['caps']:
            cap.set(linewidth=2)
        ax3.set_title('Outlier Detection - Boxplot', fontsize=14, fontweight='bold', pad=10)
        ax3.set_ylabel('Mean Pixel Intensity', fontsize=11, fontweight='bold')
        ax3.grid(axis='y', alpha=0.3)

        # Visualization 4: Color Channel Distribution
        print("Creating visualization 4/7: Color Channel Analysis...")
        ax4 = plt.subplot(3, 3, 4)
        red_mean = np.mean(self.images[:, :, :, 0])
        green_mean = np.mean(self.images[:, :, :, 1])
        blue_mean = np.mean(self.images[:, :, :, 2])

        channels = ['Red', 'Green', 'Blue']
        means = [red_mean, green_mean, blue_mean]
        colors_rgb = ['#FF6B6B', '#95E1D3', '#4A90E2']

        bars = ax4.bar(channels, means, color=colors_rgb, edgecolor='black', linewidth=1.5)
        ax4.set_title('Average Color Channel Intensity', fontsize=14, fontweight='bold', pad=10)
        ax4.set_ylabel('Mean Intensity', fontsize=11, fontweight='bold')

        for bar in bars:
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}',
                    ha='center', va='bottom', fontweight='bold')

        # Visualization 5: Class Balance Pie Chart
        print("Creating visualization 5/7: Class Balance Analysis...")
        ax5 = plt.subplot(3, 3, 5)
        colors_pie = plt.cm.Pastel1(np.linspace(0, 1, len(unique)))
        wedges, texts, autotexts = ax5.pie(counts, labels=[self.class_names[i] for i in unique],
                                           autopct='%1.1f%%', startangle=90,
                                           colors=colors_pie, explode=[0.05]*len(unique),
                                           textprops={'fontweight': 'bold'})
        ax5.set_title('Class Balance Percentage', fontsize=14, fontweight='bold', pad=10)

        # Visualization 6: Brightness vs Contrast Scatter
        print("Creating visualization 6/7: Brightness vs Contrast...")
        ax6 = plt.subplot(3, 3, 6)
        brightness = np.mean(self.images, axis=(1, 2, 3))
        contrast = np.std(self.images, axis=(1, 2, 3))
        scatter = ax6.scatter(brightness, contrast, c=self.labels, cmap='viridis',
                             alpha=0.6, edgecolors='black', s=50)
        ax6.set_title('Brightness vs Contrast Analysis', fontsize=14, fontweight='bold', pad=10)
        ax6.set_xlabel('Brightness (Mean Intensity)', fontsize=11, fontweight='bold')
        ax6.set_ylabel('Contrast (Std Dev)', fontsize=11, fontweight='bold')
        cbar = plt.colorbar(scatter, ax=ax6)
        cbar.set_label('Class', fontweight='bold')
        ax6.grid(alpha=0.3)

        # Visualization 7: Sample Images Grid
        print("Creating visualization 7/7: Sample Images...")
        sample_indices = np.random.choice(len(self.images), min(9, len(self.images)), replace=False)

        for i, idx in enumerate(sample_indices[:9]):
            ax = plt.subplot(3, 3, 7 + i)
            ax.imshow(self.images[idx])
            ax.set_title(f'{self.class_names[self.labels[idx]]}',
                        fontsize=10, fontweight='bold')
            ax.axis('off')

        plt.suptitle('Medical Image Dataset Analysis',
                    fontsize=18, fontweight='bold', y=0.995)
        plt.tight_layout(rect=[0, 0, 1, 0.99])

        # Save visualization
        viz_path = 'medical_image_analysis.png'
        plt.savefig(viz_path, dpi=300, bbox_inches='tight')
        print(f"\nVisualization saved: {viz_path}")
        plt.show()

        print("All 7 visualizations created successfully!")

        return fig

    def handle_class_imbalance(self):
        """
        STEP 6: CHECK IMBALANCE AND HANDLE IT
        Applies resampling techniques to balance dataset
        """
        print("\n" + "="*70)
        print("STEP 6: HANDLING CLASS IMBALANCE")
        print("="*70)

        unique, counts = np.unique(self.labels, return_counts=True)
        total = len(self.labels)

        print("Current Class Distribution:")
        for cls, count in zip(unique, counts):
            percentage = (count / total) * 100
            print(f"  {self.class_names[cls]:20s}: {count:5d} images ({percentage:5.2f}%)")

        # Check imbalance ratio
        max_count = max(counts)
        min_count = min(counts)
        imbalance_ratio = max_count / min_count

        print(f"\nImbalance Analysis:")
        print(f"  Largest Class: {max_count} images")
        print(f"  Smallest Class: {min_count} images")
        print(f"  Imbalance Ratio: {imbalance_ratio:.2f}")

        # Determine if rebalancing is needed (using 70/30 rule)
        max_percentage = (max_count / total) * 100
        min_percentage = (min_count / total) * 100

        is_imbalanced = max_percentage > 70 or min_percentage < 30

        if is_imbalanced or imbalance_ratio > 1.5:
            print(f"\nDataset is IMBALANCED. Applying resampling techniques...")

            # Flatten images for SMOTE
            n_samples = self.images.shape[0]
            images_flattened = self.images.reshape(n_samples, -1)

            print("Applying SMOTE (Synthetic Minority Over-sampling Technique)...")
            try:
                smote = SMOTE(random_state=42, k_neighbors=min(5, min_count-1))
                images_resampled, labels_resampled = smote.fit_resample(images_flattened, self.labels)

                # Reshape back to image format
                self.images = images_resampled.reshape(-1, *self.img_size, 3)
                self.labels = labels_resampled

                # Ensure pixel values are in valid range
                self.images = np.clip(self.images, 0, 255).astype(np.uint8)

                print(f"SMOTE applied successfully!")

            except Exception as e:
                print(f"SMOTE failed: {e}")
                print("Trying ADASYN instead...")
                try:
                    adasyn = ADASYN(random_state=42, n_neighbors=min(5, min_count-1))
                    images_resampled, labels_resampled = adasyn.fit_resample(images_flattened, self.labels)

                    self.images = images_resampled.reshape(-1, *self.img_size, 3)
                    self.labels = labels_resampled
                    self.images = np.clip(self.images, 0, 255).astype(np.uint8)

                    print("ADASYN applied successfully!")
                except Exception as e2:
                    print(f"ADASYN also failed: {e2}")
                    print("Proceeding with original imbalanced dataset.")
                    print("Will use class weights during training instead.")

            # Show new distribution
            print("\nClass Distribution After Resampling:")
            unique_new, counts_new = np.unique(self.labels, return_counts=True)
            total_new = len(self.labels)
            for cls, count in zip(unique_new, counts_new):
                percentage = (count / total_new) * 100
                print(f"  {self.class_names[cls]:20s}: {count:5d} images ({percentage:5.2f}%)")

        else:
            print("\nDataset is BALANCED. No resampling needed.")

        self.data_info['is_balanced'] = not is_imbalanced

        return self.images, self.labels

    def normalize_data(self):
        """
        Normalize pixel values to [0, 1] range
        """
        print("\nNormalizing image data to [0, 1] range...")
        self.images = self.images.astype('float32') / 255.0
        print(f"Normalization complete. New range: [{np.min(self.images):.2f}, {np.max(self.images):.2f}]")
        return self.images

# ============================================================================
# DEEP LEARNING MODEL ARCHITECTURE
# ============================================================================

class CancerDetectionNetwork:
    """
    Advanced neural network for medical image classification
    """

    def __init__(self, input_shape=(224, 224, 3), num_classes=2):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.model = None
        self.training_history = None

    def build_custom_architecture(self):
        """
        Build custom CNN with advanced regularization techniques
        """
        print("\nConstructing Deep Neural Network Architecture...")

        model = models.Sequential([
            # Input layer
            layers.Input(shape=self.input_shape),

            # Convolutional Block 1
            layers.Conv2D(32, (3, 3), padding='same'),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.Conv2D(32, (3, 3), padding='same'),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),

            # Convolutional Block 2
            layers.Conv2D(64, (3, 3), padding='same'),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.Conv2D(64, (3, 3), padding='same'),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),

            # Convolutional Block 3
            layers.Conv2D(128, (3, 3), padding='same'),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.Conv2D(128, (3, 3), padding='same'),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.4),

            # Convolutional Block 4
            layers.Conv2D(256, (3, 3), padding='same'),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.Conv2D(256, (3, 3), padding='same'),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.4),

            # Dense Layers
            layers.Flatten(),
            layers.Dense(512, kernel_regularizer=keras.regularizers.l2(0.001)),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.Dropout(0.5),

            layers.Dense(256, kernel_regularizer=keras.regularizers.l2(0.001)),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.Dropout(0.5),

            # Output layer
            layers.Dense(self.num_classes, activation='softmax')
        ])

        self.model = model

        print("\nArchitecture Summary:")
        total_params = model.count_params()
        print(f"  Total Parameters: {total_params:,}")

        return model

    def build_transfer_learning_model(self, architecture='VGG16'):
        """
        Build model using pre-trained networks
        """
        print(f"\nBuilding Transfer Learning Model with {architecture}...")

        if architecture == 'VGG16':
            base_model = VGG16(weights='imagenet', include_top=False,
                              input_shape=self.input_shape)
        elif architecture == 'ResNet50':
            base_model = ResNet50(weights='imagenet', include_top=False,
                                 input_shape=self.input_shape)
        else:
            raise ValueError(f"Unknown architecture: {architecture}")

        # Freeze base model layers
        base_model.trainable = False

        model = models.Sequential([
            base_model,
            layers.GlobalAveragePooling2D(),
            layers.Dense(512, kernel_regularizer=keras.regularizers.l2(0.001)),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.Dropout(0.5),
            layers.Dense(256, kernel_regularizer=keras.regularizers.l2(0.001)),
            layers.BatchNormalization(),
            layers.Activation('relu'),
            layers.Dropout(0.5),
            layers.Dense(self.num_classes, activation='softmax')
        ])

        self.model = model

        print(f"\nTransfer Learning Model Created:")
        print(f"  Base Model: {architecture}")
        print(f"  Trainable Parameters: {sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]):,}")

        return model

    def compile_network(self, learning_rate=0.001):
        """
        Compile model with optimizer and loss function
        """
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

        self.model.compile(
            optimizer=optimizer,
            loss='sparse_categorical_crossentropy',
            metrics=[
                'accuracy',
                keras.metrics.Precision(name='precision'),
                keras.metrics.Recall(name='recall'),
                keras.metrics.AUC(name='auc')
            ]
        )

        print("\nModel Compilation Complete")
        print(f"  Optimizer: Adam (lr={learning_rate})")
        print(f"  Loss Function: Sparse Categorical Crossentropy")
        print(f"  Metrics: Accuracy, Precision, Recall, AUC")

    def train_network(self, X_train, y_train, X_val, y_val, epochs=50, batch_size=32):
        """
        STEP 7: ADDRESS OVERFITTING AND UNDERFITTING
        Train model with advanced techniques to prevent overfitting
        """
        print("\n" + "="*70)
        print("STEP 7: MODEL TRAINING WITH OVERFITTING PREVENTION")
        print("="*70)

        # Data Augmentation to reduce overfitting
        print("\nConfiguring Data Augmentation...")
        datagen = ImageDataGenerator(
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            horizontal_flip=True,
            zoom_range=0.2,
            shear_range=0.2,
            fill_mode='nearest'
        )

        # Callbacks for overfitting prevention
        print("Setting up Training Callbacks...")

        callbacks = [
            # Early Stopping - stops training when validation loss stops improving
            EarlyStopping(
                monitor='val_loss',
                patience=15,
                restore_best_weights=True,
                verbose=1,
                mode='min'
            ),

            # Model Checkpoint - saves best model
            ModelCheckpoint(
                'best_medical_model.h5',
                monitor='val_accuracy',
                save_best_only=True,
                verbose=1,
                mode='max'
            ),

            # Reduce Learning Rate - reduces LR when plateau is reached
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=7,
                min_lr=1e-7,
                verbose=1,
                mode='min'
            )
        ]

        print("\nOverfitting Prevention Techniques Applied:")
        print("  ✓ Data Augmentation (rotation, shift, flip, zoom)")
        print("  ✓ Dropout Layers (0.25 - 0.5)")
        print("  ✓ L2 Regularization (0.001)")
        print("  ✓ Batch Normalization")
        print("  ✓ Early Stopping (patience=15)")
        print("  ✓ Learning Rate Reduction (factor=0.5)")

        print(f"\nTraining Configuration:")
        print(f"  Training Samples: {len(X_train)}")
        print(f"  Validation Samples: {len(X_val)}")
        print(f"  Batch Size: {batch_size}")
        print(f"  Maximum Epochs: {epochs}")

        print("\nStarting Training Process...")
        print("="*70)

        # Train model
        self.training_history = self.model.fit(
            datagen.flow(X_train, y_train, batch_size=batch_size),
            steps_per_epoch=len(X_train) // batch_size,
            validation_data=(X_val, y_val),
            epochs=epochs,
            callbacks=callbacks,
            verbose=1
        )

        print("\n" + "="*70)
        print("Training Complete!")
        print("="*70)

        return self.training_history

    def plot_training_analysis(self):
        """
        Create comprehensive training analysis plots
        """
        print("\nGenerating Training Analysis Visualizations...")

        history = self.training_history.history
        epochs_range = range(1, len(history['loss']) + 1)

        fig = plt.figure(figsize=(20, 12))

        # Loss curves
        ax1 = plt.subplot(2, 3, 1)
        ax1.plot(epochs_range, history['loss'], 'b-', linewidth=2, label='Training Loss')
        ax1.plot(epochs_range, history['val_loss'], 'r-', linewidth=2, label='Validation Loss')
        ax1.set_title('Model Loss Over Time', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Epoch', fontsize=11, fontweight='bold')
        ax1.set_ylabel('Loss', fontsize=11, fontweight='bold')
        ax1.legend(loc='upper right', fontsize=10)
        ax1.grid(True, alpha=0.3)

        # Accuracy curves
        ax2 = plt.subplot(2, 3, 2)
        ax2.plot(epochs_range, history['accuracy'], 'b-', linewidth=2, label='Training Accuracy')
        ax2.plot(epochs_range, history['val_accuracy'], 'r-', linewidth=2, label='Validation Accuracy')
        ax2.set_title('Model Accuracy Over Time', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Epoch', fontsize=11, fontweight='bold')
        ax2.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
        ax2.legend(loc='lower right', fontsize=10)
        ax2.grid(True, alpha=0.3)

        # Precision curves
        ax3 = plt.subplot(2, 3, 3)
        ax3.plot(epochs_range, history['precision'], 'b-', linewidth=2, label='Training Precision')
        ax3.plot(epochs_range, history['val_precision'], 'r-', linewidth=2, label='Validation Precision')
        ax3.set_title('Model Precision Over Time', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Epoch', fontsize=11, fontweight='bold')
        ax3.set_ylabel('Precision', fontsize=11, fontweight='bold')
        ax3.legend(loc='lower right', fontsize=10)
        ax3.grid(True, alpha=0.3)

        # Recall curves
        ax4 = plt.subplot(2, 3, 4)
        ax4.plot(epochs_range, history['recall'], 'b-', linewidth=2, label='Training Recall')
        ax4.plot(epochs_range, history['val_recall'], 'r-', linewidth=2, label='Validation Recall')
        ax4.set_title('Model Recall Over Time', fontsize=14, fontweight='bold')
        ax4.set_xlabel('Epoch', fontsize=11, fontweight='bold')
        ax4.set_ylabel('Recall', fontsize=11, fontweight='bold')
        ax4.legend(loc='lower right', fontsize=10)
        ax4.grid(True, alpha=0.3)

        # AUC curves
        ax5 = plt.subplot(2, 3, 5)
        ax5.plot(epochs_range, history['auc'], 'b-', linewidth=2, label='Training AUC')
        ax5.plot(epochs_range, history['val_auc'], 'r-', linewidth=2, label='Validation AUC')
        ax5.set_title('Model AUC Over Time', fontsize=14, fontweight='bold')
        ax5.set_xlabel('Epoch', fontsize=11, fontweight='bold')
        ax5.set_ylabel('AUC', fontsize=11, fontweight='bold')
        ax5.legend(loc='lower right', fontsize=10)
        ax5.grid(True, alpha=0.3)

        # Overfitting Analysis
        ax6 = plt.subplot(2, 3, 6)
        train_val_gap = np.array(history['loss']) - np.array(history['val_loss'])
        ax6.plot(epochs_range, train_val_gap, 'purple', linewidth=2)
        ax6.axhline(y=0, color='black', linestyle='--', linewidth=1)
        ax6.fill_between(epochs_range, train_val_gap, 0,
                        where=(train_val_gap > 0), alpha=0.3, color='red', label='Overfitting Risk')
        ax6.fill_between(epochs_range, train_val_gap, 0,
                        where=(train_val_gap <= 0), alpha=0.3, color='green', label='Good Fit')
        ax6.set_title('Overfitting Analysis', fontsize=14, fontweight='bold')
        ax6.set_xlabel('Epoch', fontsize=11, fontweight='bold')
        ax6.set_ylabel('Train-Val Loss Gap', fontsize=11, fontweight='bold')
        ax6.legend(loc='upper right', fontsize=10)
        ax6.grid(True, alpha=0.3)

        plt.suptitle('Training Performance Analysis', fontsize=18, fontweight='bold')
        plt.tight_layout()
        plt.savefig('training_analysis.png', dpi=300, bbox_inches='tight')
        print("Training analysis saved: training_analysis.png")
        plt.show()

        # Overfitting/Underfitting diagnosis
        final_train_loss = history['loss'][-1]
        final_val_loss = history['val_loss'][-1]
        final_train_acc = history['accuracy'][-1]
        final_val_acc = history['val_accuracy'][-1]

        print("\n" + "="*70)
        print("OVERFITTING/UNDERFITTING DIAGNOSIS")
        print("="*70)
        print(f"\nFinal Metrics:")
        print(f"  Training Loss:     {final_train_loss:.4f}")
        print(f"  Validation Loss:   {final_val_loss:.4f}")
        print(f"  Training Accuracy: {final_train_acc:.4f}")
        print(f"  Validation Accuracy: {final_val_acc:.4f}")

        loss_gap = abs(final_val_loss - final_train_loss)
        acc_gap = abs(final_train_acc - final_val_acc)

        print(f"\nPerformance Gaps:")
        print(f"  Loss Gap:     {loss_gap:.4f}")
        print(f"  Accuracy Gap: {acc_gap:.4f}")

        # Diagnosis
        print("\nDiagnosis:")
        if final_val_loss > final_train_loss * 1.15:
            print("  ⚠️  OVERFITTING DETECTED")
            print("  Model performs significantly better on training data")
            print("  Recommendations:")
            print("    - Increase dropout rate")
            print("    - Add more regularization")
            print("    - Collect more training data")
            print("    - Reduce model complexity")
        elif final_train_loss > 0.5 and final_train_acc < 0.7:
            print("  ⚠️  UNDERFITTING DETECTED")
            print("  Model is not learning patterns effectively")
            print("  Recommendations:")
            print("    - Increase model complexity")
            print("    - Train for more epochs")
            print("    - Reduce regularization")
            print("    - Engineer better features")
        else:
            print("  ✅ MODEL IS WELL-FITTED")
            print("  Good balance between bias and variance")

    def evaluate_performance(self, X_test, y_test, class_names):
        """
        STEP 8: EVALUATE MODEL PERFORMANCE USING METRICS
        Comprehensive evaluation with multiple metrics
        """
        print("\n" + "="*70)
        print("STEP 8: MODEL PERFORMANCE EVALUATION")
        print("="*70)

        print("\nGenerating predictions on test set...")
        y_pred_proba = self.model.predict(X_test, verbose=0)
        y_pred = np.argmax(y_pred_proba, axis=1)

        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)

        # Per-class metrics
        precision_per_class, recall_per_class, f1_per_class, support = \
            precision_recall_fscore_support(y_test, y_pred, zero_division=0)

        # Print classification report
        print("\n" + "="*70)
        print("CLASSIFICATION REPORT")
        print("="*70)
        print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

        # Print overall metrics
        print("="*70)
        print("OVERALL PERFORMANCE METRICS")
        print("="*70)
        print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
        print(f"  Precision: {precision:.4f} ({precision*100:.2f}%)")
        print(f"  Recall:    {recall:.4f} ({recall*100:.2f}%)")
        print(f"  F1-Score:  {f1:.4f} ({f1*100:.2f}%)")

        # Create comprehensive visualization
        fig = plt.figure(figsize=(20, 12))

        # 1. Confusion Matrix
        ax1 = plt.subplot(2, 3, 1)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=class_names, yticklabels=class_names,
                   cbar_kws={'label': 'Count'}, ax=ax1, linewidths=1, linecolor='black')
        ax1.set_title('Confusion Matrix', fontsize=14, fontweight='bold', pad=10)
        ax1.set_ylabel('True Label', fontsize=11, fontweight='bold')
        ax1.set_xlabel('Predicted Label', fontsize=11, fontweight='bold')

        # 2. Normalized Confusion Matrix
        ax2 = plt.subplot(2, 3, 2)
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='RdYlGn',
                   xticklabels=class_names, yticklabels=class_names,
                   cbar_kws={'label': 'Percentage'}, ax=ax2, linewidths=1, linecolor='black')
        ax2.set_title('Normalized Confusion Matrix', fontsize=14, fontweight='bold', pad=10)
        ax2.set_ylabel('True Label', fontsize=11, fontweight='bold')
        ax2.set_xlabel('Predicted Label', fontsize=11, fontweight='bold')

        # 3. ROC Curve
        ax3 = plt.subplot(2, 3, 3)
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
        for i in range(self.num_classes):
            fpr, tpr, _ = roc_curve(y_test == i, y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            ax3.plot(fpr, tpr, linewidth=2,
                    label=f'{class_names[i]} (AUC = {roc_auc:.3f})',
                    color=colors[i % len(colors)])

        ax3.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
        ax3.set_xlim([0.0, 1.0])
        ax3.set_ylim([0.0, 1.05])
        ax3.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
        ax3.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
        ax3.set_title('ROC Curve Analysis', fontsize=14, fontweight='bold', pad=10)
        ax3.legend(loc='lower right', fontsize=9)
        ax3.grid(True, alpha=0.3)

        # 4. Overall Metrics Bar Chart
        ax4 = plt.subplot(2, 3, 4)
        metrics_dict = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        }
        colors_metrics = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
        bars = ax4.bar(metrics_dict.keys(), metrics_dict.values(),
                      color=colors_metrics, edgecolor='black', linewidth=1.5)
        ax4.set_title('Overall Performance Metrics', fontsize=14, fontweight='bold', pad=10)
        ax4.set_ylabel('Score', fontsize=11, fontweight='bold')
        ax4.set_ylim([0, 1.1])
        ax4.axhline(y=0.9, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Excellent (>90%)')
        ax4.axhline(y=0.7, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label='Good (>70%)')

        for bar in bars:
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}\n({height*100:.1f}%)',
                    ha='center', va='bottom', fontweight='bold', fontsize=9)
        ax4.legend(loc='lower right', fontsize=8)
        ax4.grid(axis='y', alpha=0.3)

        # 5. Per-Class Performance
        ax5 = plt.subplot(2, 3, 5)
        x_pos = np.arange(len(class_names))
        width = 0.25

        bars1 = ax5.bar(x_pos - width, precision_per_class, width,
                       label='Precision', color='#FF6B6B', edgecolor='black')
        bars2 = ax5.bar(x_pos, recall_per_class, width,
                       label='Recall', color='#4ECDC4', edgecolor='black')
        bars3 = ax5.bar(x_pos + width, f1_per_class, width,
                       label='F1-Score', color='#45B7D1', edgecolor='black')

        ax5.set_xlabel('Class', fontsize=11, fontweight='bold')
        ax5.set_ylabel('Score', fontsize=11, fontweight='bold')
        ax5.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold', pad=10)
        ax5.set_xticks(x_pos)
        ax5.set_xticklabels(class_names)
        ax5.legend(fontsize=10)
        ax5.set_ylim([0, 1.1])
        ax5.grid(axis='y', alpha=0.3)

        # 6. Prediction Distribution
        ax6 = plt.subplot(2, 3, 6)
        prediction_counts = np.bincount(y_pred, minlength=len(class_names))
        actual_counts = np.bincount(y_test, minlength=len(class_names))

        x_pos = np.arange(len(class_names))
        width = 0.35

        bars1 = ax6.bar(x_pos - width/2, actual_counts, width,
                       label='Actual', color='#4ECDC4', edgecolor='black')
        bars2 = ax6.bar(x_pos + width/2, prediction_counts, width,
                       label='Predicted', color='#FF6B6B', edgecolor='black')

        ax6.set_xlabel('Class', fontsize=11, fontweight='bold')
        ax6.set_ylabel('Count', fontsize=11, fontweight='bold')
        ax6.set_title('Prediction vs Actual Distribution', fontsize=14, fontweight='bold', pad=10)
        ax6.set_xticks(x_pos)
        ax6.set_xticklabels(class_names)
        ax6.legend(fontsize=10)
        ax6.grid(axis='y', alpha=0.3)

        plt.suptitle('Comprehensive Model Evaluation Report',
                    fontsize=18, fontweight='bold')
        plt.tight_layout()
        plt.savefig('model_evaluation_report.png', dpi=300, bbox_inches='tight')
        print("\nEvaluation report saved: model_evaluation_report.png")
        plt.show()

        # Return metrics dictionary
        metrics_results = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'confusion_matrix': cm,
            'per_class_precision': precision_per_class,
            'per_class_recall': recall_per_class,
            'per_class_f1': f1_per_class
        }

        return metrics_results

    def save_model(self, filepath):
        """Save trained model"""
        self.model.save(filepath)
        print(f"\nModel saved successfully: {filepath}")
        print(f"File size: {os.path.getsize(filepath) / (1024**2):.2f} MB")

# ============================================================================
# MAIN EXECUTION PIPELINE
# ============================================================================

def run_cancer_detection_pipeline(data_path, cancer_type, model_type='custom'):
    """
    Complete pipeline for cancer detection
    """
    print("\n" + "="*70)
    print(f" {cancer_type.upper()} DETECTION PIPELINE")
    print("="*70)

    # Initialize processor
    processor = MedicalImageProcessor(data_path, image_dimensions=(224, 224))

    # STEP 1-6: Data Processing
    X, y = processor.load_dataset()
    X, y = processor.check_and_remove_duplicates()
    X, y = processor.handle_missing_and_corrupted()
    X, y, intensities = processor.detect_and_handle_outliers()
    processor.create_comprehensive_visualizations()
    X, y = processor.handle_class_imbalance()
    X = processor.normalize_data()

    # Split dataset
    print("\n" + "="*70)
    print("SPLITTING DATASET")
    print("="*70)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    print(f"\nDataset Split:")
    print(f"  Training:   {len(X_train):5d} images ({len(X_train)/len(X)*100:.1f}%)")
    print(f"  Validation: {len(X_val):5d} images ({len(X_val)/len(X)*100:.1f}%)")
    print(f"  Testing:    {len(X_test):5d} images ({len(X_test)/len(X)*100:.1f}%)")

    # Build and train model
    print("\n" + "="*70)
    print("BUILDING NEURAL NETWORK")
    print("="*70)

    model = CancerDetectionNetwork(
        input_shape=(224, 224, 3),
        num_classes=len(processor.class_names)
    )

    if model_type == 'custom':
        model.build_custom_architecture()
    else:
        model.build_transfer_learning_model(model_type)

    model.compile_network(learning_rate=0.0001)

    # STEP 7: Train with overfitting prevention
    history = model.train_network(X_train, y_train, X_val, y_val,
                                  epochs=50, batch_size=32)

    model.plot_training_analysis()

    # STEP 8: Evaluate performance
    metrics = model.evaluate_performance(X_test, y_test, processor.class_names)

    # Save model
    model_filename = f'{cancer_type.lower().replace(" ", "_")}_model.h5'
    model.save_model(model_filename)

    print("\n" + "="*70)
    print(f"{cancer_type.upper()} DETECTION PIPELINE COMPLETED")
    print("="*70)

    return model, metrics, processor.class_names

# ============================================================================
# RUN FOR BOTH CANCER TYPES
# ============================================================================

if __name__ == "__main__":

    # Configuration
    BREAST_CANCER_PATH = "data/breast_cancer/train"
    BRAIN_TUMOR_PATH = "data/brain_tumor/train"

    print("\n" + "="*70)
    print(" MEDICAL IMAGE ANALYSIS SYSTEM")
    print(" Cancer Detection Using Deep Learning")
    print("="*70)

    # Check if data directories exist
    breast_exists = os.path.exists(BREAST_CANCER_PATH)
    brain_exists = os.path.exists(BRAIN_TUMOR_PATH)

    results = {}

    # Run Breast Cancer Detection
    if breast_exists:
        try:
            print("\n\n" + "🔬"*35)
            print(" STARTING BREAST CANCER DETECTION")
            print("🔬"*35)

            breast_model, breast_metrics, breast_classes = run_cancer_detection_pipeline(
                BREAST_CANCER_PATH,
                "Breast Cancer",
                model_type='custom'
            )

            results['breast'] = {
                'model': breast_model,
                'metrics': breast_metrics,
                'classes': breast_classes
            }

        except Exception as e:
            print(f"\n❌ Error in breast cancer detection: {str(e)}")
            import traceback
            traceback.print_exc()
    else:
        print(f"\n⚠️  Breast cancer data not found at: {BREAST_CANCER_PATH}")

    # Run Brain Tumor Detection
    if brain_exists:
        try:
            print("\n\n" + "🧠"*35)
            print(" STARTING BRAIN TUMOR DETECTION")
            print("🧠"*35)

            brain_model, brain_metrics, brain_classes = run_cancer_detection_pipeline(
                BRAIN_TUMOR_PATH,
                "Brain Tumor",
                model_type='custom'
            )

            results['brain'] = {
                'model': brain_model,
                'metrics': brain_metrics,
                'classes': brain_classes
            }

        except Exception as e:
            print(f"\n❌ Error in brain tumor detection: {str(e)}")
            import traceback
            traceback.print_exc()
    else:
        print(f"\n⚠️  Brain tumor data not found at: {BRAIN_TUMOR_PATH}")

    # Final Comparison
    if len(results) > 0:
        print("\n\n" + "="*70)
        print(" FINAL COMPARATIVE ANALYSIS")
        print("="*70)

        comparison_data = []
        for cancer_type, data in results.items():
            metrics = data['metrics']
            comparison_data.append({
                'Cancer Type': cancer_type.replace('_', ' ').title(),
                'Accuracy': f"{metrics['accuracy']:.4f}",
                'Precision': f"{metrics['precision']:.4f}",
                'Recall': f"{metrics['recall']:.4f}",
                'F1-Score': f"{metrics['f1_score']:.4f}"
            })

        comparison_df = pd.DataFrame(comparison_data)
        print("\n" + comparison_df.to_string(index=False))

        # Create comparison visualization
        if len(results) == 2:
            fig, ax = plt.subplots(figsize=(12, 6))

            metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
            breast_values = [results['breast']['metrics'][m.lower().replace('-', '_')]
                           for m in metrics_names]
            brain_values = [results['brain']['metrics'][m.lower().replace('-', '_')]
                          for m in metrics_names]

            x = np.arange(len(metrics_names))
            width = 0.35

            bars1 = ax.bar(x - width/2, breast_values, width, label='Breast Cancer',
                          color='#FF6B6B', edgecolor='black')
            bars2 = ax.bar(x + width/2, brain_values, width, label='Brain Tumor',
                          color='#4ECDC4', edgecolor='black')

            ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
            ax.set_ylabel('Score', fontsize=12, fontweight='bold')
            ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels(metrics_names)
            ax.legend(fontsize=11)
            ax.set_ylim([0, 1.1])
            ax.grid(axis='y', alpha=0.3)

            # Add value labels on bars
            for bars in [bars1, bars2]:
                for bar in bars:
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height,
                           f'{height:.3f}',
                           ha='center', va='bottom', fontweight='bold', fontsize=9)

            plt.tight_layout()
            plt.savefig('final_comparison.png', dpi=300, bbox_inches='tight')
            print("\nComparison chart saved: final_comparison.png")
            plt.show()

    print("\n" + "="*70)
    print(" ✅ ALL PROCESSES COMPLETED SUCCESSFULLY")
    print("="*70)
    print("\nGenerated Files:")
    print("  📊 medical_image_analysis.png - Dataset visualizations")
    print("  📈 training_analysis.png - Training performance")
    print("  📋 model_evaluation_report.png - Evaluation metrics")
    if len(results) == 2:
        print("  📊 final_comparison.png - Model comparison")
    print("  💾 *_model.h5 - Trained models")

    print("\n" + "="*70)

 MEDICAL IMAGE ANALYSIS FOR CANCER DETECTION
 Deep Learning Based Diagnostic Support System

 MEDICAL IMAGE ANALYSIS SYSTEM
 Cancer Detection Using Deep Learning

⚠️  Breast cancer data not found at: data/breast_cancer/train

⚠️  Brain tumor data not found at: data/brain_tumor/train

 ✅ ALL PROCESSES COMPLETED SUCCESSFULLY

Generated Files:
  📊 medical_image_analysis.png - Dataset visualizations
  📈 training_analysis.png - Training performance
  📋 model_evaluation_report.png - Evaluation metrics
  💾 *_model.h5 - Trained models

